## Market Returns Data
Fetch market returns from Alpha Vantage for both equities (SPX) and bonds (AGG):


In [ ]:
ALPHA_VANTAGE_API_KEY = '1JYCUCSEKO7IYQKU'  # Alpha Vantage API key

def get_monthly_market_returns(symbol):
    """Fetch market returns from Alpha Vantage"""
    base_url = 'https://www.alphavantage.co/query'
    
    try:
        # For equities (SPX), use TIME_SERIES_DAILY_ADJUSTED
        # For ETFs (AGG), use TIME_SERIES_DAILY
        function = 'TIME_SERIES_DAILY' 
        
        params = {
            'function': function,
            'symbol': symbol,
            'apikey': ALPHA_VANTAGE_API_KEY,
            'outputsize': 'full'
        }
        
        response = requests.get(base_url, params=params)
        data = response.json()
        
        # Parse time series data
        time_series_key = 'Time Series (Daily)'
        if time_series_key not in data:
            raise ValueError(f'No data found for {symbol}')
            
        # Convert to DataFrame
        df = pd.DataFrame.from_dict(data[time_series_key], orient='index')
        df.index = pd.to_datetime(df.index)
        
        # Calculate returns
        df['return'] = df['4. close'].astype(float).pct_change()

        #calculate monthly returns
        df['return'] = df['return'].resample('M').agg(lambda x: (1 + x).prod() - 1)    
            
        return df['return'].dropna()
        
    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        return None

# Cache market returns
market_returns = {
    'EQUITY': get_monthly_market_returns('SPY'),  # SPDR S&P 500 ETF
    'BONDS': get_monthly_market_returns('AGG')    # iShares Core U.S. Aggregate Bond ETF
}
market_returns

SyntaxError: invalid syntax (3597644502.py, line 2)

## Beta Calculation and ETF Categorization
Define functions to calculate beta and categorize ETFs by yield characteristics:

In [ ]:



def analyze_betas(df):
    """Analyze beta distribution"""
    # Ensure required columns exist
    if 'ticker' not in df.columns:
        df['Ticker'] = None
    else:
        df = df.rename(columns={'ticker': 'Ticker'})
    if 'name' not in df.columns:
        df['Name'] = None
    else:
        df = df.rename(columns={'name': 'Name'})
    if 'beta' not in df.columns:
        df['Beta'] = None
    if 'Asset_Class' not in df.columns:
        df['Asset_Class'] = None

    # Add volatility and market data
    if 'last_year_volatility' in df.columns:
        df['Volatility'] = df['last_year_volatility']
    else:
        df['Volatility'] = None

    if 'last_year' in df.columns:
        df['Return'] = df['last_year']
    else:
        df['Return'] = None
    
    for asset_class in ['EQUITY', 'BONDS']:
        if asset_class in market_returns:
            mkt = market_returns[asset_class]
            mask = df['Asset_Class'].str.upper() == asset_class
            df.loc[mask, 'Market_Volatility'] = mkt.std() * np.sqrt(252)
            df.loc[mask, 'Market_Return'] = mkt.mean() * 252

    # Display results
    beta_summary = df[[
        'Ticker', 'Name', 'Asset_Class', 'Beta',
        'Volatility', 'Market_Volatility', 'Return', 'Market_Return'
    ]].copy()
    
    display(beta_summary)

In [ ]:
def calculate_beta(returns, asset_class):
    """Calculate beta using appropriate market benchmark"""
    try:
        # Convert to uppercase for matching
        asset_class = asset_class.upper()
        
        # Check for valid inputs
        if not isinstance(returns, pd.Series) or returns is None:
            return None
            
        if asset_class not in market_returns:
            return None

        # Get market returns over same period
        mkt_returns = market_returns[asset_class]
        print(f"Calculating beta for {asset_class} with {len(returns)} returns and {len(mkt_returns)} market returns")
        
        # Calculate covariance and variance
        covar = returns.cov(mkt_returns)
        print(f"Covariance: {covar}")
        market_var = mkt_returns.var()
        print(f"Market Variance: {market_var}")

        if market_var == 0:
            return None

        beta = covar / market_var
        print(f"Beta: {beta}")
        return beta
        
    except Exception as e:
        print(f"Beta calculation error: {e}")
        return None